In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from scipy.stats import pearsonr

import shap
%matplotlib inline
warnings.filterwarnings('ignore')

In [2]:
BASE_DIR = "../"
DATA_PATH = os.path.join(BASE_DIR, "data", "cancer_data_eng.csv")
IMG_DIR = os.path.join(BASE_DIR, "img")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
RANDOM_STATE = 42
N_SAMPLES = 10000

In [3]:
print("\n[0] 加载数据...")
df = pd.read_csv(DATA_PATH, low_memory=False, encoding='latin-1')
df['target'] = df['Status.Vital'].map({'VIVO': 1, 'MORTO': 0})
df = df.dropna(subset=['target'])
np.random.seed(RANDOM_STATE)
if len(df) > N_SAMPLES:
    idx = np.random.choice(len(df), N_SAMPLES, replace=False)
    df = df.iloc[idx].copy()

feature_cols = ['Age', 'year', 'Code.Profession', 'Diagnostic.means',
                'Extension', 'Raca.Color']
df_feat = df[feature_cols + ['target']].copy()

cat_cols = ['Diagnostic.means', 'Extension', 'Raca.Color']
for col in cat_cols:
    le = LabelEncoder()
    non_null = df_feat[col].dropna().astype(str)
    le.fit(non_null)
    mc = non_null.value_counts().index[0]
    def encode(x, le=le, mc=mc):
        if pd.isna(x): return np.nan
        xs = str(x)
        return le.transform([xs])[0] if xs in le.classes_ else le.transform([mc])[0]
    df_feat[col] = df_feat[col].apply(encode)

X = df_feat[feature_cols].astype(float).values
y = df_feat['target'].values
feature_names = np.array(feature_cols)

print(f"    总样本: {len(X):,}  VIVO: {y.sum():,} ({y.mean()*100:.2f}%)")

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)

imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
X_tr_imp = imputer.fit_transform(X_tr)
X_te_imp = imputer.transform(X_te)
X_tr_final = scaler.fit_transform(X_tr_imp)
X_te_final = scaler.transform(X_te_imp)


[0] 加载数据...
    总样本: 10,000  VIVO: 4,123.0 (41.23%)


In [4]:
print("\n[1] 训练 Random Forest + SHAP TreeExplainer...")
model = RandomForestClassifier(
    n_estimators=200, max_depth=8, class_weight='balanced',
    random_state=RANDOM_STATE, n_jobs=-1)
model.fit(X_tr_final, y_tr)
auc = roc_auc_score(y_te, model.predict_proba(X_te_final)[:, 1])
print(f"    RF AUC = {auc:.4f}")


[1] 训练 Random Forest + SHAP TreeExplainer...
    RF AUC = 0.9104


In [5]:
X_shap = X_te_final[:]  # 整个测试集
y_shap = y_te[:]

explainer = shap.TreeExplainer(model)
sv_full = explainer.shap_values(X_shap)

# 基准值
expected_value = explainer.expected_value
if isinstance(expected_value, list) or isinstance(expected_value, np.ndarray) and expected_value.ndim > 0:
    base_value = expected_value[1] if len(expected_value) > 1 else expected_value[0]
else:
    base_value = float(expected_value)
print(f"    基准值 (Base Value): {base_value:.4f}")

# SHAP 值: 类别 1 (VIVO)
if isinstance(sv_full, list):
    sv = sv_full[1]
else:
    sv = sv_full
    if sv.ndim == 3:
        sv = sv[:, :, 1]

# 预测概率
y_prob = model.predict_proba(X_shap)[:, 1]

    基准值 (Base Value): 0.5002


In [6]:
print("\n[2] 选择代表性样本...")

def find_representative_samples(probs, quantiles=[0.05, 0.50, 0.95]):
    """找到概率接近指定分位数的样本索引"""
    indices = []
    for q in quantiles:
        target = np.quantile(probs, q)
        idx = np.argmin(np.abs(probs - target))
        # 确保不重复
        while idx in indices and len(np.unique(probs)) > len(indices):
            probs[idx] = np.inf
            idx = np.argmin(np.abs(probs - target))
        indices.append(idx)
    return indices

# 为每个类别找到代表性样本
vivo_mask = y_shap == 1
morto_mask = y_shap == 0

sample_configs = []

if vivo_mask.sum() >= 3:
    vivo_probs = y_prob[vivo_mask]
    vivo_idx = np.where(vivo_mask)[0]
    vivo_rep = find_representative_samples(vivo_probs, [0.2, 0.5, 0.8])
    vivo_abs_idx = [vivo_idx[i] for i in vivo_rep]
    for i, abs_idx in enumerate(vivo_abs_idx):
        q_map = {0: "低(存活)", 1: "中(存活)", 2: "高(存活)"}
        sample_configs.append({
            'label': q_map[i],
            'idx': abs_idx,
            'prob': y_prob[abs_idx],
            'actual': y_shap[abs_idx]
        })

if morto_mask.sum() >= 3:
    morto_probs = y_prob[morto_mask]
    morto_idx = np.where(morto_mask)[0]
    # 对死亡类, 找概率低/中的
    morto_rep = find_representative_samples(morto_probs, [0.1, 0.3])
    morto_abs_idx = [morto_idx[i] for i in morto_rep]
    for i, abs_idx in enumerate(morto_abs_idx):
        q_map_m = {0: "低(死亡)", 1: "中(死亡)"}
        sample_configs.append({
            'label': q_map_m[i],
            'idx': abs_idx,
            'prob': y_prob[abs_idx],
            'actual': y_shap[abs_idx]
        })

# 确保只有5个
sample_configs = sample_configs[:5]

# 按概率排序
sample_configs.sort(key=lambda x: x['prob'])

print(f"\n    {'#':<4} {'Label':<15} {'Probability':>12} {'Actual':>8} {'Patient ID':>10}")
print(f"    {'-'*4} {'-'*15} {'-'*12} {'-'*8} {'-'*10}")
for i, cfg in enumerate(sample_configs):
    print(f"    {i+1:<4} {cfg['label']:<15} {cfg['prob']:>12.4f} {int(cfg['actual']):>8} {cfg['idx']:>10}")


[2] 选择代表性样本...

    #    Label            Probability   Actual Patient ID
    ---- --------------- ------------ -------- ----------
    1    低(死亡)                 0.0030        0       1315
    2    中(死亡)                 0.0132        0        315
    3    低(存活)                 0.6057        1       2866
    4    中(存活)                 0.8169        1        142
    5    高(存活)                 0.8966        1        405


In [7]:

print(f"\n[3] 生成决策路径分析图...")

N_SAMPLES_SHOW = len(sample_configs)
top_k = 8  # 每个样本展示 Top 8 特征

fig = plt.figure(figsize=(20, 5 * N_SAMPLES_SHOW))
# 创建非极坐标轴用于列1和列2
gs = fig.add_gridspec(N_SAMPLES_SHOW, 3)
axes_path = [fig.add_subplot(gs[row, 0]) for row in range(N_SAMPLES_SHOW)]
axes_bar = [fig.add_subplot(gs[row, 1]) for row in range(N_SAMPLES_SHOW)]
axes_radar = [fig.add_subplot(gs[row, 2], projection='polar') for row in range(N_SAMPLES_SHOW)]

for row, cfg in enumerate(sample_configs):
    sidx = cfg['idx']
    sample_shap = sv[sidx]
    sample_feat = X_shap[sidx]
    prob = cfg['prob']

    # 按 |SHAP| 排序
    sorted_idx = np.argsort(np.abs(sample_shap))[::-1][:top_k]

    # === 列 1: 决策路径图 ===
    ax_path = axes_path[row]

    cum_shap = np.zeros(top_k + 1)
    cum_shap[0] = 0  # 从 0 开始 (SHAP 值以基准值为中心)
    for i, idx in enumerate(sorted_idx):
        cum_shap[i + 1] = cum_shap[i] + sample_shap[idx]

    n_steps = len(sorted_idx)
    path_color = '#2ecc71' if cfg['actual'] == 1 else '#e74c3c'

    ax_path.plot(range(n_steps + 1), cum_shap[:n_steps + 1], 'o-',
                 linewidth=2.5, markersize=8, color=path_color, alpha=0.8,
                 markerfacecolor='white', markeredgecolor=path_color, markeredgewidth=1.5)

    # 填充背景
    ax_path.fill_between(range(n_steps + 1), cum_shap[:n_steps + 1], alpha=0.1, color=path_color)

    # 标注每个步骤
    for i, idx in enumerate(sorted_idx):
        fn = feature_names[idx]
        fv = sample_feat[idx]
        shap_v = sample_shap[idx]
        offset_y = 15 if i % 2 == 0 else -20
        arrow_color = '#27ae60' if shap_v > 0 else '#e74c3c'
        ax_path.annotate(
            f'{fn}\n({fv:.2f}, {shap_v:+.3f})',
            xy=(i + 1, cum_shap[i + 1]),
            xytext=(i + 1, cum_shap[i + 1]),
            fontsize=7, ha='center', va='bottom' if shap_v > 0 else 'top',
            color='#2c3e50', fontweight='bold')

    # 基准值 + 预测值标注
    ax_path.axhline(y=0, color='gray', linestyle=':', alpha=0.4)
    ax_path.axhline(y=np.log(prob / (1 - prob + 1e-10)),
                    color='blue', linestyle='--', linewidth=1, alpha=0.7)
    ax_path.text(n_steps * 0.7, np.log(prob / (1 - prob + 1e-10)),
                 f'Log-odds ≈ {np.log(prob/(1-prob+1e-10)):.2f}',
                 fontsize=8, color='blue')

    ax_path.set_xlabel('Feature Accumulation Step', fontsize=10)
    ax_path.set_ylabel('Cumulative SHAP (log-odds)', fontsize=10)
    ax_path.set_title(f'Sample {cfg["label"]} — Decision Path\n'
                      f'P(VIVO)={prob:.4f}, Actual={"VIVO" if cfg["actual"]==1 else "MORTO"}',
                      fontsize=11, fontweight='bold')
    ax_path.set_xticks(range(n_steps + 1))
    ax_path.set_xticklabels(['Base'] + [feature_names[i][:12] for i in sorted_idx],
                            rotation=30, ha='right', fontsize=7.5)
    ax_path.grid(True, alpha=0.2)

    # === 列 2: SHAP 贡献条形图 ===
    ax_bar = axes_bar[row]

    sorted_features = [feature_names[i] for i in sorted_idx]
    sorted_shap_vals = sample_shap[sorted_idx]
    bar_colors = ['#27ae60' if s > 0 else '#e74c3c' for s in sorted_shap_vals]

    bars = ax_bar.barh(range(len(sorted_features)), sorted_shap_vals,
                       color=bar_colors, alpha=0.75, edgecolor='white')

    ax_bar.set_yticks(range(len(sorted_features)))
    ax_bar.set_yticklabels([f[:18] for f in sorted_features], fontsize=9)
    ax_bar.set_xlabel('SHAP Value (log-odds change)', fontsize=10)
    ax_bar.set_title('Feature Contribution', fontsize=11, fontweight='bold')
    ax_bar.axvline(x=0, color='black', linewidth=0.5)

    # 数值标注
    for bar, val in zip(bars, sorted_shap_vals):
        offset = 0.01 if val >= 0 else -0.01
        ha = 'left' if val >= 0 else 'right'
        ax_bar.text(val + offset, bar.get_y() + bar.get_height()/2,
                    f'{val:+.3f}', ha=ha, va='center', fontsize=8)

    ax_bar.set_xlim(
        min(sorted_shap_vals) - 0.05,
        max(sorted_shap_vals) + 0.05)

    # === 列 3: 特征值雷达图 ===
    ax_radar = axes_radar[row]
    ax_radar.set_theta_offset(np.pi / 2)
    ax_radar.set_theta_direction(-1)

    angles = np.linspace(0, 2 * np.pi, len(sorted_idx), endpoint=False).tolist()
    angles += angles[:1]

    # 归一化特征值
    global_min = X_shap[:, sorted_idx].min(axis=0)
    global_max = X_shap[:, sorted_idx].max(axis=0)
    feature_vals_norm = (sample_feat[sorted_idx] - global_min) / (global_max - global_min + 1e-8)
    feature_vals_norm = np.clip(feature_vals_norm, 0, 1)
    vals_plot = np.concatenate([feature_vals_norm, [feature_vals_norm[0]]])

    ax_radar.plot(angles, vals_plot, 'o-', linewidth=2, color=path_color, alpha=0.8)
    ax_radar.fill(angles, vals_plot, alpha=0.2, color=path_color)

    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels([f'{feature_names[i]}\n({sample_feat[i]:.2f})'
                               for i in sorted_idx], fontsize=7.5)
    ax_radar.set_ylim(0, 1.1)
    ax_radar.set_yticks([0.25, 0.5, 0.75])
    ax_radar.set_yticklabels(['0.25', '0.5', '0.75'], fontsize=6)
    ax_radar.set_title('Feature Values (Normalized)', fontsize=11, fontweight='bold')
    ax_radar.grid(True, alpha=0.3)

plt.suptitle(f'SHAP Decision Path Analysis — {len(sample_configs)} Representative Samples\n'
             f'Base Value = {base_value:.4f} | Model: Random Forest | AUC = {auc:.4f}',
             fontsize=14, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "20a_shap_decision_path.png"), dpi=150, bbox_inches='tight')
plt.close()
print("  [图] 20a_shap_decision_path.png 已保存")



[3] 生成决策路径分析图...
  [图] 20a_shap_decision_path.png 已保存
